In [11]:
import pickle
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

with open('quantized_matrix_1148.pkl', 'rb') as f:
    quantized_matrix_1148 = pickle.load(f)


with open('disease_regulons.pkl', 'rb') as f:
    disease_regulons = pickle.load(f)

with open('validated.pkl', 'rb') as f:
    validated = pickle.load(f)

with open('cox_results.pkl', 'rb') as f:
    cox_results = pickle.load(f)

print(quantized_matrix_1148.shape)  # (899, 437)
print(len(disease_regulons))        # 148
print(len(validated))               # 899

(1148, 437)
148
899


In [13]:
X = quantized_matrix_1148.values.astype(float)

sil_scores = []
k_range = range(2, 61)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels)
    sil_scores.append(sil)
    print(f"k={k}: silhouette={sil:.4f}")

best_k = k_range[np.argmax(sil_scores)]
print(f"\nBest k for programs: {best_k}")

k=2: silhouette=0.1370
k=3: silhouette=0.1391
k=4: silhouette=0.1328
k=5: silhouette=0.1291
k=6: silhouette=0.1358
k=7: silhouette=0.1353
k=8: silhouette=0.1323
k=9: silhouette=0.1366
k=10: silhouette=0.1295
k=11: silhouette=0.1446
k=12: silhouette=0.0653
k=13: silhouette=0.0447
k=14: silhouette=0.1354
k=15: silhouette=0.0888
k=16: silhouette=0.0780
k=17: silhouette=0.0906
k=18: silhouette=0.0819
k=19: silhouette=0.0945
k=20: silhouette=0.0926
k=21: silhouette=0.0998
k=22: silhouette=0.0951
k=23: silhouette=0.0965
k=24: silhouette=0.0885
k=25: silhouette=0.0990
k=26: silhouette=0.0978
k=27: silhouette=0.0165
k=28: silhouette=0.1009
k=29: silhouette=0.1018
k=30: silhouette=0.1010
k=31: silhouette=0.0185
k=32: silhouette=0.0230
k=33: silhouette=0.0195
k=34: silhouette=0.0425
k=35: silhouette=0.0971
k=36: silhouette=0.0235
k=37: silhouette=0.0246
k=38: silhouette=0.0225
k=39: silhouette=0.1024
k=40: silhouette=0.0216
k=41: silhouette=0.1021
k=42: silhouette=0.0799
k=43: silhouette=0.0270


In [15]:

for k in [20, 30, 40, 50]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    program_series_temp = pd.Series(labels, index=quantized_matrix_1148.index)
    largest = program_series_temp.value_counts().iloc[0]
    sil = silhouette_score(X, labels)
    print(f"k={k}: largest cluster={largest}, silhouette={sil:.4f}")

k=20: largest cluster=388, silhouette=0.0926
k=30: largest cluster=378, silhouette=0.1010
k=40: largest cluster=305, silhouette=0.0216
k=50: largest cluster=320, silhouette=0.0226


In [16]:
# Keep only regulons with meaningful activity variation
row_std = quantized_matrix_1148.std(axis=1)
print(f"Std distribution:\n{row_std.describe()}")

# Keep top 50% most variable regulons
threshold = row_std.median()
active_regulons = row_std[row_std > threshold].index
print(f"Active regulons: {len(active_regulons)}")

X_active = quantized_matrix_1148.loc[active_regulons].values.astype(float)

for k in [11, 20, 30]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_active)
    largest = pd.Series(labels).value_counts().iloc[0]
    sil = silhouette_score(X_active, labels)
    print(f"k={k}: largest cluster={largest}, silhouette={sil:.4f}")

Std distribution:
count    1148.000000
mean        0.538557
std         0.222459
min         0.067573
25%         0.336922
50%         0.605245
75%         0.717759
max         0.978148
dtype: float64
Active regulons: 574
k=11: largest cluster=63, silhouette=0.0907
k=20: largest cluster=45, silhouette=0.0823
k=30: largest cluster=35, silhouette=0.0719


In [17]:
km_final = KMeans(n_clusters=11, random_state=42, n_init=10)
program_labels = km_final.fit_predict(X_active)
program_series = pd.Series(program_labels, index=active_regulons)

print("Regulons per program:")
print(program_series.value_counts().sort_index())

# Disease-relevant regulons per program
dr_set = set(disease_regulons)
for prog in range(11):
    regs_in_prog = program_series[program_series == prog].index.tolist()
    dr_in_prog = [r for r in regs_in_prog if r in dr_set]
    print(f"Program {prog}: {len(regs_in_prog)} regulons, {len(dr_in_prog)} disease-relevant")

Regulons per program:
0     50
1     53
2     58
3     60
4     46
5     46
6     59
7     52
8     36
9     51
10    63
Name: count, dtype: int64
Program 0: 50 regulons, 6 disease-relevant
Program 1: 53 regulons, 10 disease-relevant
Program 2: 58 regulons, 2 disease-relevant
Program 3: 60 regulons, 7 disease-relevant
Program 4: 46 regulons, 7 disease-relevant
Program 5: 46 regulons, 2 disease-relevant
Program 6: 59 regulons, 9 disease-relevant
Program 7: 52 regulons, 6 disease-relevant
Program 8: 36 regulons, 4 disease-relevant
Program 9: 51 regulons, 8 disease-relevant
Program 10: 63 regulons, 10 disease-relevant


In [18]:
# Compute program activity per patient
# For each program, mean quantized activity across its regulons per patient

program_activity = pd.DataFrame(index=range(11), columns=quantized_matrix_1148.columns)

for prog in range(11):
    regs_in_prog = program_series[program_series == prog].index.tolist()
    program_activity.loc[prog] = quantized_matrix_1148.loc[regs_in_prog].mean(axis=0)

program_activity = program_activity.astype(float)
print(f"Program activity matrix: {program_activity.shape}")  # 11 × 437

# Cluster patients into states
X_patients = program_activity.T.values  # 437 × 11

sil_scores = []
k_range = range(2, 31)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_patients)
    sil = silhouette_score(X_patients, labels)
    sil_scores.append(sil)
    print(f"k={k}: silhouette={sil:.4f}")

best_k = k_range[np.argmax(sil_scores)]
print(f"\nBest k for states: {best_k}")

Program activity matrix: (11, 437)
k=2: silhouette=0.1907
k=3: silhouette=0.1627
k=4: silhouette=0.1514
k=5: silhouette=0.1519
k=6: silhouette=0.1585
k=7: silhouette=0.1537
k=8: silhouette=0.1515
k=9: silhouette=0.1481
k=10: silhouette=0.1464
k=11: silhouette=0.1495
k=12: silhouette=0.1431
k=13: silhouette=0.1508
k=14: silhouette=0.1463
k=15: silhouette=0.1451
k=16: silhouette=0.1451
k=17: silhouette=0.1408
k=18: silhouette=0.1361
k=19: silhouette=0.1347
k=20: silhouette=0.1390
k=21: silhouette=0.1384
k=22: silhouette=0.1368
k=23: silhouette=0.1360
k=24: silhouette=0.1347
k=25: silhouette=0.1339
k=26: silhouette=0.1385
k=27: silhouette=0.1342
k=28: silhouette=0.1377
k=29: silhouette=0.1346
k=30: silhouette=0.1367

Best k for states: 2


In [19]:
km_states = KMeans(n_clusters=2, random_state=42, n_init=10)
state_labels = km_states.fit_predict(X_patients)
state_series = pd.Series(state_labels, index=program_activity.columns)

print("Patients per state:")
print(state_series.value_counts())

with open('state_labels.pkl', 'wb') as f:
    pickle.dump(state_series, f)

with open('program_series.pkl', 'wb') as f:
    pickle.dump(program_series, f)

with open('program_activity.pkl', 'wb') as f:
    pickle.dump(program_activity, f)

print("Saved.")

Patients per state:
1    266
0    171
Name: count, dtype: int64
Saved.
